# Notebook 06 — SHAP Explainability
**EPPS–American Airlines Data Challenge — GROW 26.2**

Uses SHAP (SHapley Additive exPlanations) to explain XGBoost predictions:
- Which features drive sequence risk the most?
- How does each feature affect individual predictions?
- What makes a specific airport pair risky?

All plots saved to `../outputs/figures/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import os, warnings, pickle
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
shap.initjs()

PROC_PATH    = '../data/processed/'
MODELS_PATH  = '../outputs/models/'
FIGURES_PATH = '../outputs/figures/'
RESULTS_PATH = '../outputs/results/'
os.makedirs(FIGURES_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

print('✅ Imports OK')

## 1 — Load Data & Model

In [ ]:
df = pd.read_csv(f'{PROC_PATH}sequences_scored.csv', parse_dates=['Date'])

xgb_model = XGBClassifier()
xgb_model.load_model(f'{MODELS_PATH}xgboost_final.json')

ID_COLS  = ['airport_A','airport_B','Date','delayed_A','delayed_B',
            'is_risky','risk_score_xgb','risk_score_lr','predicted_risky_xgb']
OBJ_COLS = [c for c in df.columns if df[c].dtype == 'object']
FEAT_COLS = [c for c in df.columns if c not in ID_COLS + OBJ_COLS]

X = df[FEAT_COLS].fillna(0).astype(float)
y = df['is_risky'].astype(int)

print(f'Sequences : {X.shape}')
print(f'Features  : {len(FEAT_COLS)}')

## 2 — Compute SHAP Values

In [ ]:
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X)

print(f'SHAP values shape : {shap_values.shape}')
print(f'Expected value    : {explainer.expected_value:.4f}')

## 3 — Summary Plot (Global Feature Importance)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
shap.summary_plot(shap_values, X, plot_type='bar',
                  max_display=20, show=False)
plt.title('SHAP Feature Importance — Top 20 (Bar)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}24_shap_importance_bar.png', bbox_inches='tight')
plt.show()
print('💾 24_shap_importance_bar.png')

## 4 — SHAP Beeswarm Plot (Impact Direction)

In [ ]:
plt.figure(figsize=(10, 9))
shap.summary_plot(shap_values, X, max_display=20, show=False)
plt.title('SHAP Beeswarm — Feature Impact on Risk Score',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}25_shap_beeswarm.png', bbox_inches='tight')
plt.show()
print('💾 25_shap_beeswarm.png')
print()
print('Interpretation:')
print('  Red dot = high feature value')
print('  Blue dot = low feature value')
print('  Right side (+) = increases risk prediction')
print('  Left side (-)  = decreases risk prediction')

## 5 — Waterfall Plot: Most Risky Sequence

In [ ]:
# Find the sequence with the highest risk score
most_risky_idx = df['risk_score_xgb'].idxmax()
most_risky_row = df.loc[most_risky_idx]

print('=== MOST RISKY SEQUENCE ===')
print(f'  Airport A     : {most_risky_row["airport_A"]}')
print(f'  Airport B     : {most_risky_row["airport_B"]}')
print(f'  Date          : {most_risky_row["Date"].date()}')
print(f'  Risk Score    : {most_risky_row["risk_score_xgb"]:.4f}')
print(f'  Actual Label  : {int(most_risky_row["is_risky"])} ({"Risky" if most_risky_row["is_risky"]==1 else "Not Risky"})')

shap_exp = shap.Explanation(
    values        = shap_values[most_risky_idx],
    base_values   = explainer.expected_value,
    data          = X.iloc[most_risky_idx].values,
    feature_names = FEAT_COLS
)

plt.figure(figsize=(12, 6))
shap.waterfall_plot(shap_exp, max_display=15, show=False)
plt.title(f'SHAP Waterfall — Most Risky Sequence\n{most_risky_row["airport_A"]} → DFW → {most_risky_row["airport_B"]}',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}26_shap_waterfall_risky.png', bbox_inches='tight')
plt.show()
print('💾 26_shap_waterfall_risky.png')

## 6 — Waterfall Plot: Least Risky Sequence

In [ ]:
least_risky_idx = df['risk_score_xgb'].idxmin()
least_risky_row = df.loc[least_risky_idx]

print('=== LEAST RISKY SEQUENCE ===')
print(f'  Airport A     : {least_risky_row["airport_A"]}')
print(f'  Airport B     : {least_risky_row["airport_B"]}')
print(f'  Date          : {least_risky_row["Date"].date()}')
print(f'  Risk Score    : {least_risky_row["risk_score_xgb"]:.4f}')
print(f'  Actual Label  : {int(least_risky_row["is_risky"])}')

shap_exp2 = shap.Explanation(
    values        = shap_values[least_risky_idx],
    base_values   = explainer.expected_value,
    data          = X.iloc[least_risky_idx].values,
    feature_names = FEAT_COLS
)

plt.figure(figsize=(12, 6))
shap.waterfall_plot(shap_exp2, max_display=15, show=False)
plt.title(f'SHAP Waterfall — Least Risky Sequence\n{least_risky_row["airport_A"]} → DFW → {least_risky_row["airport_B"]}',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}27_shap_waterfall_safe.png', bbox_inches='tight')
plt.show()
print('💾 27_shap_waterfall_safe.png')

## 7 — SHAP Dependence Plots (Top 4 Features)

In [ ]:
# Get top 4 features by mean |SHAP|
mean_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=FEAT_COLS)
top4 = mean_shap.sort_values(ascending=False).head(4).index.tolist()
print('Top 4 features by SHAP importance:', top4)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(top4):
    if feat in FEAT_COLS:
        feat_idx = FEAT_COLS.index(feat)
        shap.dependence_plot(
            feat_idx, shap_values, X,
            feature_names=FEAT_COLS,
            ax=axes[i], show=False
        )
        axes[i].set_title(f'SHAP Dependence: {feat}',
                          fontweight='bold', fontsize=10)

plt.suptitle('SHAP Dependence Plots — Top 4 Features',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}28_shap_dependence.png', bbox_inches='tight')
plt.show()
print('💾 28_shap_dependence.png')

## 8 — SHAP Mean Importance Table

In [ ]:
shap_importance = pd.DataFrame({
    'feature'       : FEAT_COLS,
    'mean_abs_shap' : np.abs(shap_values).mean(axis=0),
    'mean_shap'     : shap_values.mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

shap_importance['mean_abs_shap'] = shap_importance['mean_abs_shap'].round(5)
shap_importance['mean_shap']     = shap_importance['mean_shap'].round(5)

shap_importance.to_csv(f'{RESULTS_PATH}shap_feature_importance.csv', index=False)
print('💾 shap_feature_importance.csv saved')
print()
print('=== TOP 20 FEATURES BY SHAP IMPORTANCE ===')
print(shap_importance.head(20).to_string(index=False))
print()
print('✅ All notebooks complete!')
print()
print('Files saved:')
print('  outputs/figures/  → 28 visualization plots')
print('  outputs/results/  → risky_pairs_ranked.csv, model_metrics.txt, shap_feature_importance.csv')
print('  outputs/models/   → xgboost_final.json, baseline_lr.pkl')
print('  data/processed/   → inbound_clean.csv, outbound_clean.csv, sequences.csv, sequences_scored.csv')